# 01 — Running ALIGNN on JARVIS-DFT

**Purpose:** generate bulk-modulus predictions for every JARVIS-DFT material
that has a DFT reference value, and record whether each material was part of
the pretrained model's training data.

**Output:** `alignn_bulk_modulus_full_results.csv` — one row per material,
with the DFT value, the model prediction, and split membership.

**Runtime:** Google Colab with a T4 GPU. The full run takes about 5 minutes.
Analysis of the results happens in `03_analysis.ipynb`, not here.

> **Colab resets between sessions.** Installed packages, downloaded files and
> variables are all lost when the runtime disconnects. Every session starts by
> re-running from the top.

## 1. Environment check

Confirms we are on the Colab runtime and not the local laptop Python. The two
look identical in VS Code but are different machines with different files.

**Expect:** Python 3.12.x and `GPU: True`. If it says `GPU: False`, set
Runtime → Change runtime type → T4 GPU. Running 25,000 materials on CPU takes
hours instead of minutes.

In [1]:
import sys, torch
print(sys.version)
print("GPU:", torch.cuda.is_available())

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
GPU: True


## 2. Install

`alignn` brings the model and PyTorch; `jarvis-tools` brings the database
client. The `!` runs a shell command rather than Python. `-q` hides most of
the output.

Red warnings about version conflicts are normal. Only a line starting with
`ERROR` matters.

In [2]:
!pip install alignn jarvis-tools -q

Verify the install worked and record the versions — these belong in the
Methods section, since results can change between library versions.

In [3]:
import torch
print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

import alignn
print("alignn:", alignn.__version__)

torch: 2.14.1+cu130
GPU available: True
alignn: 2026.11.2


## 3. Load the pretrained model

`get_alignn2_model` downloads a zip from Figshare containing three files:

| file | what it is |
|---|---|
| `config.json` | the architecture and graph-building settings |
| `best_model.pt` | the trained weights — the numbers the model learned |
| `ids_train_val_test.json` | **which materials it was trained on** |

That third file is what makes this whole study possible. Without it there is
no way to tell which materials the model has already seen.

We build an empty network from `config.json`, then load the learned weights
into it. `.eval()` switches off training-only behaviour such as dropout.

**Note:** `bulk_modulus_kv_radius` is an ALIGNN 2.0 property model. The older
`get_prediction()` function does not work with it — it looks up a different
registry of force-field models and raises `KeyError`.

In [4]:
import json, torch
from alignn.pretrained import get_alignn2_model
from alignn.models.alignn_atomwise_pure import (
    ALIGNNAtomWisePure, ALIGNNAtomWisePureConfig
)
from alignn.graphs import Graph

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

paths = get_alignn2_model("bulk_modulus_kv_radius")
cfg = json.load(open(paths["config.json"]))

model = ALIGNNAtomWisePure(ALIGNNAtomWisePureConfig(**cfg["model"]))
model.load_state_dict(
    torch.load(paths["best_model.pt"], map_location=device, weights_only=False)
)
model.to(device).eval()
print("Model ready")

Using: cuda
Obtaining model bulk_modulus_kv_radius from https://ndownloader.figshare.com/files/67163351
Model ready


## 4. Load the database and the training splits

`data("dft_3d")` downloads the full JARVIS-DFT database (~1 GB, cached after
the first call).

The split file lives inside the same model zip we downloaded above, so we
fetch it directly from Figshare and read `ids_train_val_test.json` out of it.

**Expect:** ~93,900 materials, and split sizes of 15,744 / 1,968 / 1,968.

In [5]:
import requests, zipfile, io
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms

d = data("dft_3d")
print(f"JARVIS loaded: {len(d)} materials")

r = requests.get("https://ndownloader.figshare.com/files/67163351", timeout=300)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    hit = [n for n in z.namelist() if n.endswith("ids_train_val_test.json")][0]
    with z.open(hit) as f:
        ids = json.load(f)

train_ids = set(ids["id_train"])
val_ids   = set(ids["id_val"])
test_ids  = set(ids["id_test"])
seen_ids  = train_ids | val_ids | test_ids

print(f"train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)}")
print(f"seen total: {len(seen_ids)}")

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699


100%|██████████| 48.4M/48.4M [00:00<00:00, 57.1MiB/s]


Loading the zipfile...
Loading completed.
JARVIS loaded: 93902 materials
train 15744 | val 1968 | test 1968
seen total: 19680


## 5. Single-material sanity check

Before running thousands of materials, confirm the pipeline works on one.

BAs (JVASP-7630) has a DFT bulk modulus of 133.37 GPa.

**This material is inside the training set**, so close agreement proves the
*pipeline* works — it says nothing about the model's ability to generalise.

In [6]:
entry = [x for x in d if x["jid"] == "JVASP-7630"][0]
atoms = Atoms.from_dict(entry["atoms"])
print(entry["formula"], entry["bulk_modulus_kv"])

g, lg = Graph.atom_dgl_multigraph(
    atoms,
    neighbor_strategy="pure_torch",
    cutoff=float(cfg.get("cutoff", 5.0)),
    max_neighbors=int(cfg.get("max_neighbors", 12)),
    atom_features=cfg.get("atom_features", "cgcnn"),
    use_canonize=bool(cfg.get("use_canonize", False)),
)
lat = torch.tensor(atoms.lattice_mat).type(torch.get_autocast_dtype("cpu"))

with torch.no_grad():
    out = model((g.to(device), lg.to(device), lat.to(device)))
if isinstance(out, dict):
    out = out.get("out", out.get("energy", next(iter(out.values()))))
pred = float(out.detach().cpu().numpy().flatten()[0])

print("Predicted:", round(pred, 2), "GPa")
print("DFT value:", entry["bulk_modulus_kv"], "GPa")

BAs 133.37
Predicted: 138.6 GPa
DFT value: 133.37 GPa


## 6. The prediction function

Wraps the steps above so they can be called in a loop.

**How a crystal becomes a prediction:**
1. `Atoms.from_dict` — the stored dictionary becomes a crystal structure
2. `atom_dgl_multigraph` — the crystal becomes two graphs: `g`, where atoms are
   nodes and bonds within `cutoff` are edges, and `lg`, the *line graph*, where
   bonds become nodes so the model can see bond **angles**. The line graph is
   the "L" in ALIGNN and is the architecture's central idea.
3. the model runs on both graphs plus the lattice matrix

**Why `try/except`:** in ~25,000 materials, a few will have geometries that
fail graph construction. Without this, one bad structure kills the entire run.
Returning `None` lets the loop continue while still printing which material
failed and why.

`torch.no_grad()` skips gradient bookkeeping — we are predicting, not training.

In [7]:
def predict_one(entry):
    """Predict bulk modulus in GPa. Returns None if anything fails."""
    try:
        atoms = Atoms.from_dict(entry["atoms"])
        g, lg = Graph.atom_dgl_multigraph(
            atoms,
            neighbor_strategy="pure_torch",
            cutoff=float(cfg.get("cutoff", 5.0)),
            max_neighbors=int(cfg.get("max_neighbors", 12)),
            atom_features=cfg.get("atom_features", "cgcnn"),
            use_canonize=bool(cfg.get("use_canonize", False)),
        )
        lat = torch.tensor(atoms.lattice_mat).type(torch.get_autocast_dtype("cpu"))
        with torch.no_grad():
            out = model((g.to(device), lg.to(device), lat.to(device)))
        if isinstance(out, dict):
            out = out.get("out", out.get("energy", next(iter(out.values()))))
        return float(out.detach().cpu().numpy().flatten()[0])
    except Exception as e:
        print(f"  FAILED on {entry.get('jid')}: {e}")
        return None

## 7. Build the candidate lists

Keep only materials with a real numeric bulk modulus — JARVIS stores missing
values as the string `"na"`, which `isinstance(..., (int, float))` filters out.

Then split into two pools by whether the model has seen them.

**Expect:** 32,433 candidates, of which 12,753 are unseen.

> **Order matters.** This cell must run *before* the sampling cell below.
> In the original notebook these were the wrong way round, so `Run All` failed.

In [8]:
candidates = [
    e for e in d
    if isinstance(e.get("bulk_modulus_kv"), (int, float))
    and e["bulk_modulus_kv"] != "na"
]
print(f"Candidates with bulk modulus data: {len(candidates)}")

unseen_candidates = [e for e in candidates if e["jid"] not in seen_ids]
seen_candidates   = [e for e in candidates if e["jid"] in seen_ids]

print(f"unseen: {len(unseen_candidates)} | seen: {len(seen_candidates)}")

Candidates with bulk modulus data: 32433
unseen: 12753 | seen: 19680


## 8. Twenty-material trial

Never launch a long run on untested code. A bug found at material #1 costs
seconds; the same bug at material #20,000 costs an hour.

Ten from each pool so both are represented. `random.seed(42)` makes the
selection reproducible — anyone rerunning this gets the same twenty materials.
**Always seed randomness in research code.**

**Check before continuing:** do the predicted values look like real bulk moduli
(tens to hundreds of GPa), and is the `was_seen` column a mix of True and
False? If every row says True, the sampling is biased and the trial proves
nothing.

In [9]:
import random
import pandas as pd

random.seed(42)
sample = random.sample(unseen_candidates, 10) + random.sample(seen_candidates, 10)
random.shuffle(sample)

rows = []
for entry in sample:
    rows.append({
        "jid": entry["jid"],
        "formula": entry["formula"],
        "dft": entry["bulk_modulus_kv"],
        "predicted": predict_one(entry),
        "was_seen": entry["jid"] in seen_ids,
    })

print(pd.DataFrame(rows))

             jid   formula     dft   predicted  was_seen
0    JVASP-51043    Mn3GaN  227.40  227.492538      True
1    JVASP-69081    K2BaMn    7.19    7.347091      True
2   JVASP-114801    TlPdCl   38.91   40.656322      True
3   JVASP-190332    CaYZn2   49.50   44.254177     False
4     JVASP-3864  Rb2TiCl6   18.50   18.300989      True
5   JVASP-135875      BaSO   25.60   29.358955     False
6   JVASP-119919     RbBH4   40.50   32.635437      True
7   JVASP-156300  Mg2As2Pb   53.13   50.382347     False
8    JVASP-68027   Na2MnBe   29.97   33.130924      True
9   JVASP-134931     VGeO2   77.26   83.887184     False
10   JVASP-20239      FeRh  181.27  182.056107      True
11  JVASP-160952    NaBiAu   42.77   44.303585     False
12  JVASP-136944    NaLiPt   52.53   52.326408     False
13  JVASP-134047     NaBeH   27.37   28.263060     False
14  JVASP-192800  LiZrInRh  115.00  108.504799     False
15   JVASP-30438    Sb2S2O   32.44   31.838713      True
16  JVASP-149941   ThAgAsO   77

## 9. The full run

**Design:** all 12,753 unseen materials, plus an equal-sized random sample from
the seen pool. Equal sizes so neither group dominates the comparison.

`in_test_set` is recorded separately because test-split materials were **not**
trained on — they are held out by the model's authors. Mixing them into the
"seen" group would be wrong, and it is what lets us reproduce the published
benchmark later.

**Checkpointing:** the CSV is written every 500 materials. Colab can disconnect
without warning; if it dies at material 8,000 the file still holds 7,500 rows.
The progress line reports the rate and estimated time remaining.

**Expect:** ~25,506 rows, roughly 90 materials/sec on a T4, about 5 minutes.
A handful of `FAILED` lines is normal; hundreds would indicate a real problem.

In [10]:
import time

random.seed(42)
seen_sample = random.sample(seen_candidates,
                            min(len(unseen_candidates), len(seen_candidates)))

full_run_list = unseen_candidates + seen_sample
random.shuffle(full_run_list)

CHECKPOINT_EVERY = 500
OUT_PATH = "alignn_bulk_modulus_full_results.csv"

results = []
start = time.time()

for i, entry in enumerate(full_run_list):
    results.append({
        "jid": entry["jid"],
        "formula": entry["formula"],
        "dft": entry["bulk_modulus_kv"],
        "predicted": predict_one(entry),
        "was_seen": entry["jid"] in seen_ids,
        "in_test_set": entry["jid"] in test_ids,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(full_run_list) - i - 1) / rate
        print(f"{i+1}/{len(full_run_list)} | {rate:.1f}/sec | "
              f"~{remaining/60:.1f} min left | saved checkpoint")

pd.DataFrame(results).to_csv(OUT_PATH, index=False)
print("DONE. Saved to", OUT_PATH)
print(f"Total rows: {len(results)}")

500/25506 | 80.4/sec | ~5.2 min left | saved checkpoint
1000/25506 | 85.0/sec | ~4.8 min left | saved checkpoint
1500/25506 | 84.8/sec | ~4.7 min left | saved checkpoint
2000/25506 | 84.6/sec | ~4.6 min left | saved checkpoint
2500/25506 | 85.4/sec | ~4.5 min left | saved checkpoint
3000/25506 | 84.7/sec | ~4.4 min left | saved checkpoint
3500/25506 | 85.5/sec | ~4.3 min left | saved checkpoint
4000/25506 | 84.8/sec | ~4.2 min left | saved checkpoint
4500/25506 | 85.5/sec | ~4.1 min left | saved checkpoint
5000/25506 | 84.9/sec | ~4.0 min left | saved checkpoint
5500/25506 | 85.5/sec | ~3.9 min left | saved checkpoint
6000/25506 | 85.0/sec | ~3.8 min left | saved checkpoint
6500/25506 | 85.3/sec | ~3.7 min left | saved checkpoint
7000/25506 | 84.9/sec | ~3.6 min left | saved checkpoint
7500/25506 | 85.2/sec | ~3.5 min left | saved checkpoint
8000/25506 | 84.8/sec | ~3.4 min left | saved checkpoint
8500/25506 | 84.9/sec | ~3.3 min left | saved checkpoint
9000/25506 | 84.8/sec | ~3.2 min

## 10. Save off the Colab machine

**Do this immediately.** The file above lives only on Google's temporary
machine. When the session ends it is deleted.

The download saves to your browser's Downloads folder, usually with no popup.
Move it into the project's `data/` folder afterwards.

In [11]:
from google.colab import files
files.download("alignn_bulk_modulus_full_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## End of notebook

The result file is now on your laptop. All analysis — error statistics,
label-quality checks, percentiles, statistical tests and figures — happens in
`03_analysis.ipynb`, which reads the CSV and never needs Colab or a GPU.

### Removed from the original notebook

| Cell(s) | Reason |
|---|---|
| `get_prediction` attempt | wrong API generation, raised `KeyError` |
| `dir(p)` listing | debugging aid, not part of the method |
| empty `split_ids.json` write | placeholder that created an unused file |
| duplicate `groupby` cells | identical code run twice |
| `dft >= 5` cleaning cells | **superseded** — the paper uses `dft > 0`; keeping both thresholds made the notebook disagree with the Methods section |
| analysis + figure cells | moved to `03_analysis.ipynb` |
| two empty cells | — |